In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset

# Setting for generating dataset

In [2]:
# Load the network for Case5
net = {
    "baseMVA": 100.0,
## area data
    "areas": np.array([[1, 4]]),
## bus data
###	bus_i	type	Pd	Qd	Gs	Bs	area	Vm	Va	baseKV	zone	Vmax	Vmin
    "bus": np.array([
            [1, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [2, 1, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [3, 2, 300.0, 98.61, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [4, 3, 400.0, 131.47, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000],
            [5, 2, 0.0, 0.0, 0.0, 0.0, 1, 1.00000, 0.00000, 230.0, 1, 1.10000, 0.90000]
        ]),
## generator data
###	bus	Pg	Qg	Qmax	Qmin	Vg	mBase	status	Pmax	Pmin
    "gen": np.array([
            [1, 20.0,  0.0, 30.0,  -30.0,  1.0, 100.0, 1, 40.0,  0.0],
            [1, 85.0,  0.0, 127.5, -127.5, 1.0, 100.0, 1, 170.0, 0.0],
            [3, 260.0, 0.0, 390.0, -390.0, 1.0, 100.0, 1, 520.0, 0.0],
            [4, 100.0, 0.0, 150.0, -150.0, 1.0, 100.0, 1, 200.0, 0.0],
            [5, 300.0, 0.0, 450.0, -450.0, 1.0, 100.0, 1, 600.0, 0.0]
        ]),
## generator cost data
###	2	startup	shutdown	n	c(n-1)	...	c0
    "gencost": np.array([
            [2, 0.0, 0.0, 3, 0.000000, 14.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 15.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 30.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 40.000000, 0.000000],
            [2, 0.0, 0.0, 3, 0.000000, 10.000000, 0.000000]
        ]),
## branch data
###	fbus	tbus	r	x	b	rateA	rateB	rateC	ratio	angle	status	angmin	angmax
    "branch": np.array([
            [1, 2, 0.00281, 0.0281, 0.00712, 400.0, 400.0, 400.0, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 4, 0.00304, 0.0304, 0.00658, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [1, 5, 0.00064, 0.0064, 0.03126, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [2, 3, 0.00108, 0.0108, 0.01852, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [3, 4, 0.00297, 0.0297, 0.00674, 426, 426, 426, 0.0, 0.0, 1, -30.0, 30.0],
            [4, 5, 0.00297, 0.0297, 0.00674, 240.0, 240.0, 240.0, 0.0, 0.0, 1, -30.0, 30.0]
        ])
}

In [57]:
print("Type of net:", type(net))


Type of net: <class 'dict'>


In [3]:
cost_gen = pd.DataFrame(net['gencost'], columns = ['2', 'startup', 'shutdown', '3','n', 'c(n-1)', 'c0'])
cost_gen

,2,startup,shutdown,3,n,c(n-1),c0
0,2.0,0.0,0.0,3.0,0.0,14.0,0.0
1,2.0,0.0,0.0,3.0,0.0,15.0,0.0
2,2.0,0.0,0.0,3.0,0.0,30.0,0.0
3,2.0,0.0,0.0,3.0,0.0,40.0,0.0
4,2.0,0.0,0.0,3.0,0.0,10.0,0.0


In [4]:
gencost_array = cost_gen[["c(n-1)"]].to_numpy()

print("Gencost array:")
print(gencost_array)

Gencost array:
[[14.]
 [15.]
 [30.]
 [40.]
 [10.]]


In [5]:
line = pd.DataFrame(net['branch'],
                    columns = ['fbus',	'tbus',	'r',	'x',	'b',
                               'rateA',	'rateB',	'rateC',	'ratio',
                               'angle',	'status',	'angmin',	'angmax'] )
line

,fbus,tbus,r,x,b,rateA,rateB,rateC,ratio,angle,status,angmin,angmax
0,1.0,2.0,0.00281,0.0281,0.00712,400.0,400.0,400.0,0.0,0.0,1.0,-30.0,30.0
1,1.0,4.0,0.00304,0.0304,0.00658,426.0,426.0,426.0,0.0,0.0,1.0,-30.0,30.0
2,1.0,5.0,0.00064,0.0064,0.03126,426.0,426.0,426.0,0.0,0.0,1.0,-30.0,30.0
3,2.0,3.0,0.00108,0.0108,0.01852,426.0,426.0,426.0,0.0,0.0,1.0,-30.0,30.0
4,3.0,4.0,0.00297,0.0297,0.00674,426.0,426.0,426.0,0.0,0.0,1.0,-30.0,30.0
5,4.0,5.0,0.00297,0.0297,0.00674,240.0,240.0,240.0,0.0,0.0,1.0,-30.0,30.0


In [6]:
num_buses = len(net['bus'])

In [7]:
"""Initialize the admittance matrix"""

Y = np.zeros((num_buses, num_buses), dtype=complex)

# Populate the admittance matrix
for _, row in line.iterrows():
    fbus, tbus, r, x, b = int(row['fbus']) - 1, int(row['tbus']) - 1, row['r'], row['x'], row['b']
    y = 1 / (r + 1j * x)  # Line admittance
    Y[fbus, tbus] -= y
    Y[tbus, fbus] -= y  # Symmetric off-diagonal
    Y[fbus, fbus] += y + 1j * b / 2  # Diagonal element for from-bus
    Y[tbus, tbus] += y + 1j * b / 2  # Diagonal element for to-bus

# Separate into G (conductance) and B (susceptance)
G_matrix = Y.real
B_matrix = Y.imag


# Convert to DataFrames for better visualization
G_df = pd.DataFrame(G_matrix, columns=[f"Bus {i+1}" for i in range(num_buses)], index=[f"Bus {i+1}" for i in range(num_buses)])
B_df = pd.DataFrame(B_matrix, columns=[f"Bus {i+1}" for i in range(num_buses)], index=[f"Bus {i+1}" for i in range(num_buses)])

print("Conductance Matrix (G):")
print(G_df)

print("\nSusceptance Matrix (B):")
print(B_df)

Conductance Matrix (G):
           Bus 1      Bus 2      Bus 3     Bus 4      Bus 5
Bus 1  22.250686  -3.523484   0.000000 -3.256905 -15.470297
Bus 2  -3.523484  12.691067  -9.167583  0.000000   0.000000
Bus 3   0.000000  -9.167583  12.501250 -3.333667   0.000000
Bus 4  -3.256905   0.000000  -3.333667  9.924238  -3.333667
Bus 5 -15.470297   0.000000   0.000000 -3.333667  18.803964

Susceptance Matrix (B):
            Bus 1       Bus 2       Bus 3      Bus 4       Bus 5
Bus 1 -222.484377   35.234840    0.000000  32.569046  154.702970
Bus 2   35.234840 -126.897854   91.675834   0.000000    0.000000
Bus 3    0.000000   91.675834 -124.999871  33.336667    0.000000
Bus 4   32.569046    0.000000   33.336667 -99.232350   33.336667
Bus 5  154.702970    0.000000    0.000000  33.336667 -188.020637


# Ouput from MatPower

In [8]:
df = pd.read_csv('./data/pglib_opf_case5_pjm.csv')
df.head()

,load1:pl,load2:pl,load3:pl,load1:ql,load2:ql,load3:ql,gen1:pg,gen2:pg,gen3:pg,gen4:pg,...,line3:p_fr_max,line4:p_fr_max,line5:p_fr_max,line6:p_fr_max,line1:q_fr_max,line2:q_fr_max,line3:q_fr_max,line4:q_fr_max,line5:q_fr_max,line6:q_fr_max
0,6.570765,2.275993,4.051883,4.376283,0.628149,4.026378,0.4,1.7,4.876424,1.558644,...,-0.000002,-0.000003,-2.720498e-07,-0.000006,0.000000e+00,0.0,-7.933284e-07,-1.873404e-06,0.0,-3.069352e-06
1,6.552550,1.704638,4.090173,4.400894,0.724693,3.937011,0.4,1.7,4.317437,1.531758,...,-0.000002,-0.000003,-2.624331e-07,-0.000006,0.000000e+00,0.0,-7.848656e-07,-1.892890e-06,0.0,-2.936123e-06
2,5.645406,2.029723,5.955067,1.377338,0.865517,3.240306,0.4,1.7,5.199999,1.490930,...,-0.000002,-0.000002,0.000000e+00,-0.000280,-1.365476e-07,0.0,0.000000e+00,-8.479213e-07,0.0,-3.177477e-07
3,4.778645,2.417017,6.361179,0.843277,0.881531,2.635903,0.4,1.7,5.200000,1.644658,...,-0.000001,-0.000001,0.000000e+00,-0.647117,-3.141235e-07,0.0,0.000000e+00,-7.238938e-07,0.0,-6.550166e-09
4,5.690040,2.458801,5.745494,2.003454,1.119158,3.286294,0.4,1.7,5.200000,1.684051,...,-0.000002,-0.000002,-5.028490e-09,-0.000067,0.000000e+00,0.0,0.000000e+00,-9.201327e-07,0.0,-6.686726e-07


# Funtion to generate the tensor dataset based on the input data

In [9]:
def generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost, num_cost_coefficients=1):
    """
    Generate a tensor dataset following the specified format.

    Args:
        data (pd.DataFrame): DataFrame containing the sample data for loads, generation, and branch information.
        num_buses (int): Number of buses (N_B).
        num_generators (int): Number of generators (N_G).
        G_matrix (np.array): Conductance matrix
        B_matrix (np.array): Susceptance matrix
        gencost (np.array): Generator cost data matrix
        num_cost_coefficients (int): Number of cost coefficients (N_C) # 0. Default is 1 for 1st-order polynomial.

    Returns:
        np.ndarray: Tensor dataset of shape [N_I, (4 + N_C * N_G), N_B, N_B].
    """
    # Extract column mappings
    pd_columns = [col for col in data.columns if "load" in col and ":pl" in col]  # Active power loads (P_d)
    qd_columns = [col for col in data.columns if "load" in col and ":ql" in col]  # Reactive power loads (Q_d)
    num_samples = len(data)
    num_channels = 4 + (num_generators * num_cost_coefficients)  # 4 primary channels + cost channels

    # Initialize the tensor dataset
    tensor_dataset = np.zeros((num_samples, num_channels, num_buses, num_buses))

    # Process each sample
    for sample_idx in range(num_samples):
        sample = data.iloc[sample_idx]

        # Step 1: Active and Reactive Power Demand Matrices (P_d and Q_d)
        Pd_matrix = np.zeros((num_buses, num_buses))
        Qd_matrix = np.zeros((num_buses, num_buses))
        Pd_matrix[np.diag_indices(len(pd_columns))] = sample[pd_columns].values
        Qd_matrix[np.diag_indices(len(qd_columns))] = sample[qd_columns].values

        # Step 2: Placeholder Admittance Matrices (G and B)
        G_matrix = G_matrix
        B_matrix = B_matrix

        # Step 3: Cost Matrices for Generation Costs
        cost_matrices = []
        for gen_idx in range(num_generators):
            cost_matrix = np.zeros((num_buses, num_buses))
            
            # Extract cost efficients from gencost
            coefficient = gencost[gen_idx, 0]

            # Populate digonal with the generation cost coefficient
            cost_matrix[gen_idx, gen_idx] = coefficient
                
            cost_matrices.append(cost_matrix)
            
        # Combine all channels into a tensor for this sample
        sample_tensor = [Pd_matrix, Qd_matrix, G_matrix, B_matrix] + cost_matrices
        tensor_dataset[sample_idx] = np.stack(sample_tensor, axis=0)

    return tensor_dataset

In [10]:
file_path = './data/pglib_opf_case5_pjm.csv'
data = pd.read_csv(file_path)

In [11]:
num_buses = 5
num_generators = 5

In [12]:
tensor_dataset = generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix,gencost=gencost_array)

In [13]:
tensor_dataset.shape

(10000, 9, 5, 5)

In [14]:
tensor_dataset[0][4]

array([[14.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.]])

# Define dataset for training 

In [15]:
def generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost, num_cost_coefficients=1):
    """
    Generate a tensor dataset following the specified format.

    Args:
        data (pd.DataFrame): DataFrame containing the sample data for loads, generation, and branch information.
        num_buses (int): Number of buses (N_B).
        num_generators (int): Number of generators (N_G).
        G_matrix (np.array): Conductance matrix
        B_matrix (np.array): Susceptance matrix
        gencost (np.array): Generator cost data matrix
        num_cost_coefficients (int): Number of cost coefficients (N_C). Default is 1 for 1st-order polynomial.

    Returns:
        pd.DataFrame: Updated DataFrame with new calculated columns.
        np.ndarray: Tensor dataset of shape [N_I, (4 + N_C * N_G), N_B, N_B].
    """
    df = data.copy()
    
    """==================================Output============================================="""
    # Define outputs
    gen_p = [col for col in df.columns if col.endswith(":pg")]
    gen_q = [col for col in df.columns if col.endswith(":qg")]

    def convert(x):
        if isinstance(x, str):
            x = x.replace(" + ", "+").replace(" - ", "-").replace(" j ", "j").strip()
            return np.complex64(x)
        else:
            return np.complex64(0)

    bus_vm = []
    bus_va = []
    for bus_v_column in [col for col in df.columns if ":v_bus" in col]:
        df[bus_v_column + '_mag'] = df[bus_v_column].apply(convert).apply(np.abs)          # calculate the magnitude
        bus_vm.append(bus_v_column + '_mag')

        df[bus_v_column + '_ang'] = df[bus_v_column].apply(convert).apply(np.angle)        # calculate the angle
        df[bus_v_column + '_ang'] = -1 * np.rad2deg(df[bus_v_column + '_ang'].values)      # convert to radian
        bus_va.append(bus_v_column + '_ang')
        
    # Define outputs
    outputs = gen_p + gen_q + bus_vm + bus_va

    """==================================Input============================================="""
    # Extract column mappings
    pd_columns = [col for col in data.columns if "load" in col and ":pl" in col]  # Active power loads (P_d)
    qd_columns = [col for col in data.columns if "load" in col and ":ql" in col]  # Reactive power loads (Q_d)
    num_samples = len(data)
    num_channels = 4 + (num_generators * num_cost_coefficients)  # 4 primary channels + cost channels

    # Initialize the tensor dataset
    tensor_dataset = np.zeros((num_samples, num_channels, num_buses, num_buses))

    # Process each sample
    for sample_idx in range(num_samples):
        sample = data.iloc[sample_idx]

        # Step 1: Active and Reactive Power Demand Matrices (P_d and Q_d)
        Pd_matrix = np.zeros((num_buses, num_buses))
        Qd_matrix = np.zeros((num_buses, num_buses))
        Pd_matrix[np.diag_indices(len(pd_columns))] = sample[pd_columns].values
        Qd_matrix[np.diag_indices(len(qd_columns))] = sample[qd_columns].values

        # Step 2: Admittance Matrices (G and B)
        G_matrix = G_matrix
        B_matrix = B_matrix

        # Step 3: Cost Matrices for Generation Costs
        cost_matrices = []
        for gen_idx in range(num_generators):
            cost_matrix = np.zeros((num_buses, num_buses))

            # Extract cost coefficients from gencost
            coefficient = gencost[gen_idx, 0]

            # Populate diagonal with the generation cost coefficient
            cost_matrix[gen_idx, gen_idx] = coefficient

            cost_matrices.append(cost_matrix)

        # Combine all channels into a tensor for this sample
        sample_tensor = [Pd_matrix, Qd_matrix, G_matrix, B_matrix] + cost_matrices
        tensor_dataset[sample_idx] = np.stack(sample_tensor, axis=0)

    return df[outputs], tensor_dataset


In [16]:
class OPFDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_tensor = torch.tensor(self.inputs[idx]).to(torch.float32)
        output_tensor = torch.tensor(self.outputs[idx]).to(torch.float32)
        return input_tensor, output_tensor
        

In [17]:
outputs, inputs = generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost=gencost_array)


In [18]:
outputs.shape

(10000, 20)

In [19]:
inputs.shape

(10000, 9, 5, 5)

In [20]:
inputs[0][0]

array([[6.57076508, 0.        , 0.        , 0.        , 0.        ],
       [0.        , 2.27599315, 0.        , 0.        , 0.        ],
       [0.        , 0.        , 4.05188278, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ]])

In [21]:
array = inputs[0][0]

In [22]:
non_zero_values = array[array != 0]
non_zero_values

array([6.57076508, 2.27599315, 4.05188278])

In [23]:
array_1 = inputs[0][1]

In [24]:
non_zero_values_1 = array_1[array_1 != 0]
non_zero_values_1

array([4.37628285, 0.6281488 , 4.02637841])

In [25]:
a = inputs[0][0][inputs[0][0] != 0]
a

array([6.57076508, 2.27599315, 4.05188278])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(tensor_dataset, outputs.values, test_size=0.2, random_state=42)

In [27]:
train_dataset = OPFDataset(X_train, y_train)
test_dataset = OPFDataset(X_test, y_test)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Define model

In [29]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import os
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import trange, tqdm

In [30]:
class CNN(nn.Module):
    def __init__(self, channels_in, y_size):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(channels_in, 64, 3, 1, 1, padding_mode='reflect')

        self.norm = nn.LayerNorm(64)
        self.mha  = nn.MultiheadAttention(64, num_heads=1, batch_first=True)
        self.scale = nn.Parameter(torch.zeros(1))

        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(128)

        self.drop = nn.Dropout(0.5)
        self.fc_1 = nn.Linear(128*5*5, 256)
        self.fc_out = nn.Linear(256, y_size)

    def use_attention(self, x):
        bs, c, h, w = x.shape
        x_attn = x.reshape(bs, c, h * w).transpose(1, 2)    # BS X HW X C

        x_attn = self.norm(x_attn)
        att_out, att_map = self.mha(x_attn, x_attn, x_attn)
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map

    def forward(self, x):
        x = self.conv1(x)
        x = self.scale * self.use_attention(x)[0] + x
        x = F.relu(x)
        x = F.relu(self.bn1(self.conv2(x)))
        x = F.relu(self.bn2(self.conv3(x)))
        x = x.view(x.shape[0], -1)
        x = F.relu(self.fc_1(x))
        return self.fc_out(x)


        
        

In [31]:
# Set device to GPU_indx if GPU is avaliable
gpu_indx = 0
device = torch.device(gpu_indx if torch.cuda.is_available() else 'cpu')

In [32]:
dataiter = next(iter(test_loader))

test_sample, test_label = dataiter

In [33]:
model = CNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1]).to(device)

In [34]:
num_model_params = 0
for param in model.parameters():
    num_model_params += param.flatten().shape[0]

print("-This Model Has %d (Approximately %d Million) Parameters!" % (num_model_params, num_model_params//1e6))

-This Model Has 957781 (Approximately 0 Million) Parameters!


In [35]:
# Pass image through network
out = model(test_sample.to(device))
# Check output
out.shape

/home/taiht/anaconda3/lib/python3.11/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at /opt/conda/conda-bld/pytorch_1712608853085/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


torch.Size([64, 20])

In [36]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [37]:
def train(data_loader, model, loss_fn, optimizer):
    size = len(data_loader.dataset)
    model.train()
    with torch.enable_grad():
        for batch, (X,y) in enumerate(data_loader):
            X, y = X.to(device), y.to(device)

            pred = model(X)
            loss = loss_fn(pred, y)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if batch % 10 == 0:
                loss, current = loss.item(), (batch) * len(X)
                print(f"Loss {loss:>7f} [{current:>5d}/{size:>5d}]")

In [38]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    print(f"Test Error: \n Avg loss : {test_loss:>8f} \n")

In [39]:
# epochs = 100
# for t in range(epochs):
#     print(f"Epoch {t + 1}\n -------------------")
#     train(train_loader, model, loss_fn, optimizer)
#     test(test_loader, model, loss_fn)
# print("Done !!")

In [40]:
from pypower import idx_bus, idx_gen, idx_brch
from pypower.api import makeYbus, ext2int

net = ext2int(net)

In [41]:
class OPF_metric():
    def __init__(self, net):
        basemva = net['baseMVA']
        linear_cost = net['gencost'][:,5]
        self.cost_coef = torch.tensor(linear_cost).to(torch.float32)

        pg_max = net['gen'][:, idx_gen.PMAX] / basemva
        pg_min = net['gen'][:, idx_gen.PMIN] / basemva
        qg_max = net['gen'][:, idx_gen.QMAX] / basemva
        qg_min = net['gen'][:, idx_gen.QMIN] / basemva
        vm_max = net['bus'][:, idx_bus.VMAX]
        vm_min = net['bus'][:, idx_bus.VMIN]
        va_max = [np.pi/2 for i in range(len(net['bus']))]
        va_min = [-np.pi/2 for i in range(len(net['bus']))]
        outputs_min = np.concatenate([pg_min, qg_min, vm_min, va_min])
        outputs_max = np.concatenate([pg_max, qg_max, vm_max, va_max])
        self.outputs_min = torch.as_tensor(outputs_min).to(torch.float32).view(1,-1)
        self.outputs_max = torch.as_tensor(outputs_max).to(torch.float32).view(1,-1)

        # ============= Calculate the bus admittance matrix (Ybus) and branch admittance matrices (Yf, Yt) ============#
        net = ext2int(net)
        Ybus, Yf, Yt = makeYbus(net['baseMVA'], net['bus'], net['branch'])
        Ybus = Ybus.todense()
        self.Ybus_real = torch.as_tensor(Ybus.real).to(torch.float32)
        self.Ybus_imag = torch.as_tensor(Ybus.imag).to(torch.float32)
        Yf = Yf.todense()
        Yt = Yt.todense()
        self.Yf_real = torch.as_tensor(Yf.real).to(torch.float32)
        self.Yf_imag = torch.as_tensor(Yf.imag).to(torch.float32)
        self.Yt_real = torch.as_tensor(Yt.real).to(torch.float32)
        self.Yt_imag = torch.as_tensor(Yt.imag).to(torch.float32)

        self.gen_bus_index = net['gen'][:,idx_gen.GEN_BUS]
        self.load_bus_index = [i for i in range(5) if net['bus'][i, idx_bus.PD]>0]

        self.fbus = net['branch'][:,idx_brch.F_BUS].astype(int)
        self.tbus = net['branch'][:,idx_brch.T_BUS].astype(int)
        self.smax = torch.as_tensor(net['branch'][:,idx_brch.RATE_A] / basemva).to(torch.float32)
        self.angmax = torch.as_tensor(np.deg2rad(net['branch'][:,idx_brch.ANGMAX])).to(torch.float32)

    # Let's find the generation cost
    def cal_gen_cost(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        cost_coef = self.cost_coef.to(device)
        pg = outputs[:, :5]
        return (pg  * cost_coef).sum(1)

    def cal_upper_lower_bound_violation(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        outputs_max = self.outputs_max.to(device)  # Move to the same device
        outputs_min = self.outputs_min.to(device)  # Move to the same device
    
        # Use the local variables outputs_max and outputs_min
        vio_1 = torch.relu(outputs - outputs_max)
        vio_2 = torch.relu(outputs_min - outputs)
        return vio_1 + vio_2

    def gen_load_to_bus(self, inputs, outputs):
        device = outputs.device
        batch = inputs.shape[0]
        # Extract diagonal non-zero values of pd and qd from inputs
        pd = torch.stack([torch.diagonal(inputs[b, 0, :, :], dim1=-2, dim2=-1)[torch.diagonal(inputs[b, 0, :, :], dim1=-2, dim2=-1) != 0] for b in range(batch)])
        qd = torch.stack([torch.diagonal(inputs[b, 1, :, :], dim1=-2, dim2=-1)[torch.diagonal(inputs[b, 1, :, :], dim1=-2, dim2=-1) != 0] for b in range(batch)])
        pg = outputs[:, 0:5]
        qg = outputs[:, 5:10]
        bus_pg = torch.zeros([batch, 5], device=device)
        bus_qg = torch.zeros([batch, 5], device=device)
        for i, bus_index in enumerate(self.gen_bus_index):
            bus_pg[:, int(bus_index)] = bus_pg[:, int(bus_index)] + pg[:, i]
            bus_qg[:, int(bus_index)] = bus_qg[:, int(bus_index)] + qg[:, i]
        bus_pd = torch.zeros([batch, 5], device=device)
        bus_qd = torch.zeros([batch, 5], device=device)
        for i, bus_index in enumerate(self.load_bus_index):
            bus_pd[:, bus_index] = bus_pd[:, bus_index] + pd[:, i]
            bus_qd[:, bus_index] = bus_qd[:, bus_index] + qd[:, i]
        return bus_pg, bus_qg, bus_pd, bus_qd

    def cal_power_balance_violation(self, inputs, outputs):
        device = outputs.device 
        bus_pg, bus_qg, bus_pd, bus_qd = self.gen_load_to_bus(inputs, outputs)
        bus_p_inj = bus_pg - bus_pd
        bus_q_inj = bus_qg - bus_qd
        
        vm, va = outputs[:, 10:15], outputs[:, 15:20]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Ybus_real = self.Ybus_real.to(device)
        self.Ybus_imag = self.Ybus_imag.to(device)

        Ir = torch.matmul(vr, self.Ybus_real) - vi @ self.Ybus_imag
        Ii = vi @ self.Ybus_real + vr @ self.Ybus_imag
        bus_p = (vr * Ir) + (vi * Ii)
        bus_q = (vi * Ir) - (vr * Ii)
        return torch.abs(bus_p_inj - bus_p), torch.abs(bus_q_inj - bus_q)

    # Let's calculate the branch flow & angle constraint violation
    def cal_branch_flow_vio(self, outputs):
        device = outputs.device 
        vm, va = outputs[:, 10:15], outputs[:, 15:20]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Yf_real = self.Yf_real.to(device)
        self.Yf_imag = self.Yf_imag.to(device)
        self.Yt_real = self.Yt_real.to(device)
        self.Yt_imag = self.Yt_imag.to(device)
        self.smax = self.smax.to(device)
        self.angmax = self.angmax.to(device)
        
        Irf = vr @ self.Yf_real.T - vi @ self.Yf_imag.T
        Iif = vi @ self.Yf_real.T + vr @ self.Yf_imag.T
        branch_pf = vr[:, self.fbus] * Irf + vi[:, self.fbus] * Iif
        branch_qf = vi[:, self.fbus] * Irf - vr[:, self.fbus] * Iif
        sf = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        Irt = vr @ self.Yt_real.T - vi @ self.Yt_imag.T
        Iit = vi @ self.Yt_real.T + vr @ self.Yt_imag.T
        branch_pf = vr[:, self.tbus] * Irt + vi[:, self.tbus] * Iit
        branch_qf = vi[:, self.tbus] * Irt - vr[:, self.tbus] * Iit
        st = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        branch_flow = torch.maximum(sf, st)
        branch_ang = torch.abs(va[:, self.fbus] - va[:, self.tbus])
        return torch.relu(branch_flow - self.smax), torch.relu(branch_ang - self.angmax)

In [42]:
[i for i in range(5) if net['bus'][i, idx_bus.PD]>0]

[1, 2, 3]

In [43]:
opf_metric = OPF_metric(net)

In [44]:
def test(dataloader, model, opf_metric):
    device = next(model.parameters()).device
    batch_snapshot = next(iter(dataloader))
    X, Y = batch_snapshot[0].to(device), batch_snapshot[1].to(device)
    with torch.no_grad():
      Y_pred = model(X)

    test_sol_mse = ((Y_pred - Y)**2)

    Y_pred_cost = opf_metric.cal_gen_cost(Y_pred)
    Y_opt_cost = opf_metric.cal_gen_cost(Y)
    test_obj_gap = (Y_pred_cost - Y_opt_cost)/Y_opt_cost

    vio_bound = opf_metric.cal_upper_lower_bound_violation(Y_pred)
    vio_p_mis, vio_q_mis = opf_metric.cal_power_balance_violation(X, Y_pred)
    vio_flow, vio_ang = opf_metric.cal_branch_flow_vio(Y_pred)
    eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
    ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)
    return test_sol_mse, test_obj_gap, eq_vio, ineq_vio

def model_train_test(train_dataloader, test_dataloader, model,
                     loss_fn, opf_metric, epochs = 10, penalty=False):
    torch.manual_seed(42)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    results = {
        "train_losses": [],
        "test_mse": [],
        "opt_gaps": [],
        "eq_vios": [],
        "ineq_vios": []
    }
    print(f"==============================================Start training on {device}===========================================")

    for t in range(epochs):
        model.train()
        train_loss = []
        with torch.enable_grad():
            for batch, (X, y) in enumerate(train_dataloader):
                X, y = X.to(device), y.to(device)
                # Compute prediction error
                pred = model(X)
                if penalty:
                  loss = loss_fn(X, pred, y)
                else:
                  loss = loss_fn(pred, y)
                # Backpropagation
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                train_loss.append(loss.item())
            train_loss = np.mean(train_loss)
            results["train_losses"].append(train_loss)
            test_sol_mse, test_obj_gap, eq_vio, ineq_vio = test(test_dataloader, model, opf_metric)
            
            results["test_mse"].append(test_sol_mse)
            results["opt_gaps"].append(test_obj_gap)
            results["eq_vios"].append(eq_vio)
            results["ineq_vios"].append(ineq_vio)
            
            print(f"Epoch {t+1} | Train loss: {train_loss:.3f} | ",
                f"Test mse: {test_sol_mse.mean():.3f} | ",
                f"opt gap: {test_obj_gap.mean():.3f}% | ",
                f"eq vio: {eq_vio.sum(1).mean():.3f} | ",
                f"ineq vio: {ineq_vio.sum(1).mean():.3f} |")
        
    return results


In [45]:
model_base = CNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1]).to(device)

In [46]:
results = model_train_test(train_loader, test_loader, model_base, nn.MSELoss(), opf_metric, epochs=100)

==============================================Start training on cuda:0===========================================
Epoch 1 | Train loss: 0.374 |  Test mse: 0.345 |  opt gap: 0.052% |  eq vio: 165.225 |  ineq vio: 54.280 |
Epoch 2 | Train loss: 0.253 |  Test mse: 0.205 |  opt gap: 0.022% |  eq vio: 80.226 |  ineq vio: 16.626 |
Epoch 3 | Train loss: 0.138 |  Test mse: 0.124 |  opt gap: 0.122% |  eq vio: 135.213 |  ineq vio: 37.221 |
Epoch 4 | Train loss: 0.085 |  Test mse: 0.081 |  opt gap: 0.033% |  eq vio: 159.224 |  ineq vio: 52.877 |
Epoch 5 | Train loss: 0.065 |  Test mse: 0.055 |  opt gap: -0.052% |  eq vio: 91.151 |  ineq vio: 30.413 |
Epoch 6 | Train loss: 0.058 |  Test mse: 0.050 |  opt gap: -0.022% |  eq vio: 100.183 |  ineq vio: 32.659 |
Epoch 7 | Train loss: 0.048 |  Test mse: 0.054 |  opt gap: -0.026% |  eq vio: 92.222 |  ineq vio: 20.051 |
Epoch 8 | Train loss: 0.048 |  Test mse: 0.047 |  opt gap: -0.023% |  eq vio: 86.401 |  ineq vio: 17.716 |
Epoch 9 | Train loss: 0.043 | 

In [47]:
""" Let's add the violatio into loss function and minimize it
"""
class MSEPenaltyLoss(nn.Module):
    def __init__(self, opf_metric, w_eq=0.1, w_ineq=0.1):
        super().__init__()
        self.w_eq = w_eq
        self.w_ineq = w_ineq
        self.mse = nn.MSELoss()
        self.opf_metric = opf_metric

    def forward(self, X, pred, target):
        MSE = self.mse(pred, target)
        
        vio_bound = self.opf_metric.cal_upper_lower_bound_violation(pred)
        vio_p_mis, vio_q_mis = self.opf_metric.cal_power_balance_violation(X, pred)
        vio_flow, vio_ang = self.opf_metric.cal_branch_flow_vio(pred)

        eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
        ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)

        loss = MSE + self.w_eq * eq_vio.mean() + self.w_ineq * ineq_vio.mean()
        return loss

In [48]:
model_base = CNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1]).to(device)

In [49]:
results = model_train_test(train_loader, test_loader, model_base, nn.MSELoss(), opf_metric, epochs=100)

==============================================Start training on cuda:0===========================================
Epoch 1 | Train loss: 0.377 |  Test mse: 0.353 |  opt gap: 0.050% |  eq vio: 113.591 |  ineq vio: 33.197 |
Epoch 2 | Train loss: 0.245 |  Test mse: 0.193 |  opt gap: 0.054% |  eq vio: 168.705 |  ineq vio: 57.528 |
Epoch 3 | Train loss: 0.128 |  Test mse: 0.117 |  opt gap: 0.086% |  eq vio: 94.605 |  ineq vio: 23.961 |
Epoch 4 | Train loss: 0.081 |  Test mse: 0.068 |  opt gap: 0.008% |  eq vio: 76.414 |  ineq vio: 18.081 |
Epoch 5 | Train loss: 0.062 |  Test mse: 0.057 |  opt gap: -0.104% |  eq vio: 84.460 |  ineq vio: 21.763 |
Epoch 6 | Train loss: 0.054 |  Test mse: 0.049 |  opt gap: -0.032% |  eq vio: 82.356 |  ineq vio: 20.969 |
Epoch 7 | Train loss: 0.044 |  Test mse: 0.042 |  opt gap: -0.004% |  eq vio: 77.516 |  ineq vio: 20.760 |
Epoch 8 | Train loss: 0.044 |  Test mse: 0.048 |  opt gap: -0.004% |  eq vio: 77.127 |  ineq vio: 21.635 |
Epoch 9 | Train loss: 0.041 |  T

In [50]:
class BoundCNN(nn.Module):
    def __init__(self, channels_in, y_size, yl, yu):
        super(BoundCNN, self).__init__()
        self.conv1 = nn.Conv2d(channels_in, 64, 3, 1, 1, padding_mode='reflect')

        self.norm = nn.LayerNorm(64)
        self.mha  = nn.MultiheadAttention(64, num_heads=1, batch_first=True)
        self.scale = nn.Parameter(torch.zeros(1))

        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(128)

        self.drop = nn.Dropout(0.5)
        self.fc_1 = nn.Linear(128*5*5, 256)
        self.fc_out = nn.Linear(256, y_size)

        self.register_buffer('yl', yl.clone().detach().float())
        self.register_buffer('yu', yu.clone().detach().float())

    def use_attention(self, x):
        bs, c, h, w = x.shape
        x_attn = x.reshape(bs, c, h * w).transpose(1, 2)    # BS X HW X C

        x_attn = self.norm(x_attn)
        att_out, att_map = self.mha(x_attn, x_attn, x_attn)
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map

    def forward(self, x):
        x = self.conv1(x)
        x = self.scale * self.use_attention(x)[0] + x
        x = F.relu(x)
        x = F.relu(self.bn1(self.conv2(x)))
        x = F.relu(self.bn2(self.conv3(x)))
        x = x.view(x.shape[0], -1)
        x = F.relu(self.fc_1(x))
        x = self.fc_out(x)
        x = torch.sigmoid(x)
        x = x * (self.yu - self.yl) + self.yl
        return x

In [55]:
model_base = BoundCNN(channels_in=9, y_size=20, yl=opf_metric.outputs_min, yu=opf_metric.outputs_max).to(device)

In [56]:
results = model_train_test(train_loader, test_loader, model_base, MSEPenaltyLoss(opf_metric), opf_metric, epochs=1000, penalty=True)

==============================================Start training on cuda:0===========================================
Epoch 1 | Train loss: 1.376 |  Test mse: 0.341 |  opt gap: 0.078% |  eq vio: 26.283 |  ineq vio: 0.000 |
Epoch 2 | Train loss: 0.693 |  Test mse: 0.346 |  opt gap: 0.075% |  eq vio: 9.210 |  ineq vio: 0.000 |
Epoch 3 | Train loss: 0.629 |  Test mse: 0.344 |  opt gap: 0.078% |  eq vio: 25.654 |  ineq vio: 4.586 |
Epoch 4 | Train loss: 0.507 |  Test mse: 0.335 |  opt gap: 0.081% |  eq vio: 11.698 |  ineq vio: 0.004 |
Epoch 5 | Train loss: 0.576 |  Test mse: 0.336 |  opt gap: 0.087% |  eq vio: 15.190 |  ineq vio: 0.625 |
Epoch 6 | Train loss: 0.606 |  Test mse: 0.338 |  opt gap: 0.068% |  eq vio: 23.154 |  ineq vio: 5.887 |
Epoch 7 | Train loss: 0.492 |  Test mse: 0.335 |  opt gap: 0.080% |  eq vio: 17.137 |  ineq vio: 4.095 |
Epoch 8 | Train loss: 0.478 |  Test mse: 0.335 |  opt gap: 0.074% |  eq vio: 11.421 |  ineq vio: 0.872 |
Epoch 9 | Train loss: 0.473 |  Test mse: 0.329 